# Week 4 · Machine learning
Nights 22–28 · 14–20 December 2026

Each exercise is a code cell. Replace `raise NotImplementedError(...)` with your code and run the cell
with **Shift+Enter**: the checks underneath print **✅ Correct** when it works. Stuck for 20 minutes?
Compare with `solutions/week4_machine_learning_solutions.ipynb`, then retype the answer yourself.

Tonight's reading and videos are in the study guide: [Week 4 tab](https://claude.ai/artifact/5MJxvaW9sNxjp7rjmPWuCu#dc7bfe50-ffc7).

In [ ]:
# Run this cell first (Shift+Enter). It finds your project folder and defines `check`,
# which prints "Correct" when an exercise works.
import math
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data"


def _same(a, b, tol=1e-6):
    import math as _math
    import numbers as _numbers
    try:
        import numpy as _np
    except ImportError:
        _np = None
    if _np is not None and (isinstance(a, _np.ndarray) or isinstance(b, _np.ndarray)):
        a, b = _np.asarray(a), _np.asarray(b)
        if a.shape != b.shape:
            return False
        if a.dtype.kind in "biuf" and b.dtype.kind in "biuf":
            return bool(_np.allclose(a, b, rtol=1e-5, atol=tol))
        return bool((a == b).all())
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if isinstance(a, _numbers.Number) and isinstance(b, _numbers.Number):
        return _math.isclose(a, b, rel_tol=1e-5, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def check(label, actual, expected=True, tol=1e-6):
    """Run `actual` (usually a lambda) and print Correct if it equals `expected`."""
    try:
        got = actual() if callable(actual) else actual
    except NotImplementedError:
        print(f"⏳ Not done yet: {label}")
        return
    except Exception as error:  # show the problem without stopping the notebook
        print(f"❌ {label}: your code raised {type(error).__name__}: {error}")
        return
    if _same(got, expected, tol):
        print(f"✅ Correct: {label}")
    else:
        print(f"❌ {label}\n   expected: {expected!r}\n   got:      {got!r}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

print("Ready")

## Night 22 · Mon 14 Dec: Your first scikit-learn model
Every scikit-learn model has `fit` and `predict`. Always score on rows the model never saw while training.

In [ ]:
penguins = pd.read_csv(DATA / "penguins.csv").dropna()
X_pen = penguins[["flipper_length_mm"]]  # a table of inputs (2-D), even with one column
y_pen = penguins["body_mass_g"]          # the value to predict (1-D)
print(X_pen.shape, y_pen.shape)

In [ ]:
# 22.1 Split, fit, score
def split(X, y):
    """Split into 80% train / 20% test with train_test_split(..., test_size=0.2, random_state=42).
    Return X_train, X_test, y_train, y_test."""
    raise NotImplementedError("Your code here")


def fit_and_score(X_train, X_test, y_train, y_test):
    """Fit LinearRegression on the training rows. Return (model, mean absolute error on the test rows)."""
    raise NotImplementedError("Your code here")

check("266 training rows", lambda: len(split(X_pen, y_pen)[0]), 266)
check("67 test rows", lambda: len(split(X_pen, y_pen)[1]), 67)
check("test MAE is about 290 g", lambda: round(float(fit_and_score(*split(X_pen, y_pen))[1]), 1), 290.1, tol=0.05)

In [ ]:
try:
    model, test_mae = fit_and_score(*split(X_pen, y_pen))
    print(f"Each extra mm of flipper adds about {model.coef_[0]:.1f} g.")
    print(f"On penguins it never saw, the model is off by {test_mae:.0f} g on average.")
    print("Your Night 21 model was off by about 313 g, measured on the same rows it trained on.")
except NotImplementedError:
    print("Finish split() and fit_and_score() first.")

## Night 23 · Tue 15 Dec: Classification and its metrics
The sigmoid turns a score into a probability. Accuracy misleads when one class is rare; the threshold trades
precision against recall.

In [ ]:
titanic = pd.read_csv(DATA / "titanic.csv")
data = titanic[["survived", "pclass", "sex", "age", "fare"]].dropna()
data = data.assign(is_female=(data["sex"] == "female").astype(int))
X = data[["pclass", "is_female", "age", "fare"]]
y = data["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "training rows,", len(X_test), "test rows")

In [ ]:
# 23.1 A classifier and its confusion matrix, by hand
def train_classifier(X_train, y_train):
    """Return LogisticRegression(max_iter=1000) fitted on the training rows."""
    raise NotImplementedError("Your code here")


def confusion_counts(y_true, y_pred):
    """Count by hand: {'tp': ..., 'fp': ..., 'tn': ..., 'fn': ...} (1 = survived is the positive class)."""
    raise NotImplementedError("Your code here")


def precision_recall(counts):
    """Return (precision, recall): precision = tp / (tp + fp), recall = tp / (tp + fn)."""
    raise NotImplementedError("Your code here")


def predict_with_threshold(model, X, threshold):
    """Return 1 where model.predict_proba(X)[:, 1] >= threshold, else 0, as a NumPy array."""
    raise NotImplementedError("Your code here")

check("confusion_counts on a tiny example", lambda: confusion_counts([1, 0, 1, 1, 0], [1, 1, 0, 1, 0]), {"tp": 2, "fp": 1, "tn": 1, "fn": 1})
check("precision_recall", lambda: precision_recall({"tp": 2, "fp": 1, "tn": 1, "fn": 1}), (2 / 3, 2 / 3))
check("test accuracy", lambda: round(accuracy_score(y_test, train_classifier(X_train, y_train).predict(X_test)), 3), 0.804)
check(
    "your counts on the test set",
    lambda: confusion_counts(y_test.tolist(), train_classifier(X_train, y_train).predict(X_test).tolist()),
    {"tp": 44, "fp": 14, "tn": 71, "fn": 14},
)
check("a 0.3 threshold raises recall to 0.828", lambda: round(recall_score(y_test, predict_with_threshold(train_classifier(X_train, y_train), X_test, 0.3)), 3), 0.828)

## Night 24 · Wed 16 Dec: Overfitting and cross-validation
Memorizing is not learning. Validation chooses settings; the test set judges once, at the end. Cross-validation
averages over several splits, so one lucky split can't fool you.

In [ ]:
# 24.1 Deeper trees: training accuracy keeps rising, test accuracy doesn't
def depth_scores(X_train, y_train, X_test, y_test, depths=range(1, 16)):
    """For each depth, fit DecisionTreeClassifier(max_depth=depth, random_state=42) on the training rows.
    Return a DataFrame with columns depth, train_acc and test_acc (use model.score)."""
    raise NotImplementedError("Your code here")


def best_depth_by_cv(X, y, depths=range(1, 16)):
    """Return the depth whose DecisionTreeClassifier(max_depth=depth, random_state=42) has the highest
    mean cross_val_score(..., cv=5) accuracy."""
    raise NotImplementedError("Your code here")

check("15 depths, 3 columns", lambda: depth_scores(X_train, y_train, X_test, y_test).shape, (15, 3))
check(
    "depth 15 overfits: train beats test by over 15 points",
    lambda: float(depth_scores(X_train, y_train, X_test, y_test).iloc[-1]["train_acc"] - depth_scores(X_train, y_train, X_test, y_test).iloc[-1]["test_acc"]) > 0.15,
    True,
)
check("best depth by 5-fold cross-validation", lambda: best_depth_by_cv(X, y), 5)

In [ ]:
# 24.2 Plot train and test accuracy against depth
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: scores = depth_scores(...); plot train_acc and test_acc against depth, with a legend
plt.show()

## Night 25 · Thu 17 Dec: Decision trees and random forests
A forest averages many trees, so it overfits less than one deep tree. Feature importance shows what the model leans on.

In [ ]:
# 25.1 Compare three models with cross-validation
def compare_models(X, y):
    """Return a dict: name -> mean 5-fold cross_val_score accuracy, for
    'logistic': LogisticRegression(max_iter=1000),
    'tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'forest': RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)."""
    raise NotImplementedError("Your code here")


def feature_importances(X, y):
    """Fit the same forest on all of X, y. Return a dict: feature -> importance, highest first."""
    raise NotImplementedError("Your code here")

check("cross-validated accuracy", lambda: compare_models(X, y), {"logistic": 0.789, "tree": 0.790, "forest": 0.815}, tol=0.005)
check("the forest leans most on is_female", lambda: list(feature_importances(X, y))[0], "is_female")

In [ ]:
# 25.2 Bar chart of the feature importances
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: pd.Series(feature_importances(X, y)).plot.barh(ax=ax), then a title
plt.show()

## Night 26 · Fri 18 Dec: Preparing data: missing values, categories, scaling
Models only take numbers. Fill the gaps, one-hot encode the categories, scale the numbers, and fit every
transformer on the training rows only.

In [ ]:
numeric = ["age", "fare", "sibsp", "parch"]
categorical = ["sex", "pclass", "embarked"]
X_all = titanic[numeric + categorical]
y_all = titanic["survived"]
X_all.isna().sum()

In [ ]:
# 26.1 One ColumnTransformer for all the preparation
def make_preprocessor():
    """Return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical),
    ])"""
    raise NotImplementedError("Your code here")

def _prepared():
    result = make_preprocessor().fit_transform(X_all)
    return result.toarray() if hasattr(result, "toarray") else result


check("4 numbers + 2 sexes + 3 classes + 3 ports = 12 columns", lambda: _prepared().shape, (891, 12))
check("no missing values left", lambda: int(np.isnan(_prepared()).sum()), 0)

In [ ]:
try:
    print(make_preprocessor().fit(X_all).get_feature_names_out())
except NotImplementedError:
    print("Finish make_preprocessor() first.")

## Night 27 · Sat 19 Dec: Pipelines and data leakage
A pipeline keeps preparation and model together, so cross-validation re-fits the preparation inside every fold.
Leakage lets information from the test rows into training, and the scores come out too good to be true.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print(len(X_train), "training rows,", len(X_test), "test rows")

In [ ]:
# 27.1 A pipeline, cross-validated, then tuned with a grid search
def make_model():
    """Return make_pipeline(make_preprocessor(), LogisticRegression(max_iter=1000))."""
    raise NotImplementedError("Your code here")


def tune(model, X, y):
    """Run GridSearchCV(model, grid, cv=5) on X, y with
    grid = {"logisticregression__C": [0.1, 1.0, 10.0], "logisticregression__class_weight": [None, "balanced"]}.
    Return (best_params_, best_score_)."""
    raise NotImplementedError("Your code here")

check("pipeline steps", lambda: list(make_model().named_steps), ["columntransformer", "logisticregression"])
check("5-fold CV accuracy on the training rows", lambda: round(float(cross_val_score(make_model(), X_train, y_train, cv=5).mean()), 3), 0.798, tol=0.002)
check(
    "best settings",
    lambda: tune(make_model(), X_train, y_train)[0],
    {"logisticregression__C": 10.0, "logisticregression__class_weight": None},
)

### 27.2 Catch a leak
The table below is pure noise: 100 rows, 5,000 random columns and random labels, so no model can honestly beat 50%.
The leaky version picks the 20 columns most related to the labels using *all* rows, then cross-validates.
The honest version does the picking inside a pipeline, so each fold only sees its own training rows.
Scaling or filling gaps before splitting leaks in the same way, just less visibly.

In [ ]:
def leaky_vs_honest():
    """Return (leaky_score, honest_score), both mean 5-fold cross_val_score accuracies.
    Data: rng = np.random.default_rng(0); X = rng.normal(size=(100, 5000)); y = rng.integers(0, 2, size=100).
    Leaky: X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y), then cross-validate LogisticRegression(max_iter=1000) on X_sel.
    Honest: cross-validate make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)) on X."""
    raise NotImplementedError("Your code here")

check("leaky vs honest", lambda: leaky_vs_honest(), (0.85, 0.48), tol=0.01)

## Night 28 · Sun 20 Dec: Month 1 project: Titanic end to end
1. Compare three models, each in a pipeline with `make_preprocessor()`, using 5-fold cross-validation on the
   **training rows only**.
2. Fit the best one on all training rows and score it **once** on the test rows.
3. Print a results table and paste it into your README under *Month 1 results*. Commit and push.

In [ ]:
# 28.1 Your month 1 project
models = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "decision tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "random forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
}
# TODO 1: results = {name: mean 5-fold CV accuracy of make_pipeline(make_preprocessor(), model) on X_train, y_train}
# TODO 2: best_name = the name with the highest score
# TODO 3: final = that pipeline fitted on X_train, y_train; test_accuracy = final.score(X_test, y_test)
# TODO 4: print a markdown table of the results, plus the test accuracy

### 28.2 What did you learn this month?
Write three sentences for your README: what worked, what surprised you, what you'd try next.

### 28.3 Optional: your first Kaggle submission
Join the [Titanic competition](https://www.kaggle.com/competitions/titanic), download `train.csv` and `test.csv`
from its Data page into `data/`, and run the cell below. Then upload `submission.csv` on the competition's Submit page.

In [ ]:
kaggle_train, kaggle_test = DATA / "train.csv", DATA / "test.csv"
if kaggle_train.exists() and kaggle_test.exists():
    train = pd.read_csv(kaggle_train).rename(columns=str.lower)  # Kaggle's columns are capitalized
    test = pd.read_csv(kaggle_test).rename(columns=str.lower)
    model = make_pipeline(make_preprocessor(), models[best_name]).fit(train[numeric + categorical], train["survived"])
    submission = pd.DataFrame({"PassengerId": test["passengerid"], "Survived": model.predict(test[numeric + categorical])})
    submission.to_csv(ROOT / "submission.csv", index=False)
    print(f"Saved submission.csv with {len(submission)} rows. Upload it on the competition's Submit page.")
else:
    print("Optional: put Kaggle's train.csv and test.csv in data/ first.")